# 256 × 256 tile evaluation

Set **DATASET** in the next cell and run all cells. The notebook scores the saved
tile event streams against the corresponding crop of the recorded ground truth.
It caches per-tile scores and writes CSVs and interactive HTML to the report folder
printed below. Input events come from `data_test_exp/<dataset>_256/<split>/`.

In [ ]:
from pathlib import Path

DATASET = "bsergb"           # eds, hqevfi, bsergb
SPLIT = "test"
SEQUENCES = None             # None = every sequence found; otherwise a list of folder names
METHODS = None               # None = every saved method; otherwise e.g. ["evtween_v7_student", "v2e"]
DATA_ROOT = None             # None = source in input.json, then EVTWEEN_DATA / benchmark default
PRED_ROOT = None             # None = <repo>/data_test_exp
REPORT_ROOT = None           # None = reports/experiments/<dataset>_256/<split>/watch_img_size/runs

TABLE_METRICS = None         # None = all metrics supported by this notebook
VOXEL_BIN_MS = 10.0
MATCH_TILES = True           # compare methods on common (sequence, tile) pairs within each metric
SCORE_WORKERS = 1            # parallel sequences; raise only if RAM allows concurrent full-sequence PMSE grids
CHAMFER_THREADS = 4

PLOT_METHOD = None           # None = method with the lowest valid per-tile Chamfer
PLOT_SEQUENCE = None         # optionally restrict the search to one sequence
PLOT_TILE = None             # optionally restrict the search to e.g. "tile_0003"
PLOT_START_MS = None         # None = busiest GT window for space-time and 3D views
PLOT_WINDOW_MS = 40.0
RATE_BIN_MS = 10.0
MAX_PLOT_POINTS = 15_000     # cap per displayed trace; metrics use the full streams

In [ ]:
import sys

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import HTML, display
from plotly.subplots import make_subplots

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "experiments" / "score_image_tiles.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from the Evtween repository.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from benchmarks import event_rate_views_v2 as rates
from experiments import score_image_tiles as scoring

metrics = tuple(dict.fromkeys(("chamfer_full", *(TABLE_METRICS or scoring.METRICS))))
records, methods, report = scoring.evaluate(
    DATASET, SPLIT, methods=METHODS, sequences=SEQUENCES, metrics=metrics,
    data_root=DATA_ROOT, pred_root=PRED_ROOT, report_root=REPORT_ROOT,
    voxel_bin_ms=VOXEL_BIN_MS, workers=SCORE_WORKERS, chamfer_threads=CHAMFER_THREADS,
)
raw = pd.DataFrame(records)
raw["value"] = pd.to_numeric(raw["value"], errors="coerce")
valid = raw[raw.status.isin(scoring.VALID_STATUS) & np.isfinite(raw.value)].copy()
print("Report folder:", report)

failed = raw[raw.status == "error"]
if not failed.empty:
    display(failed[["sequence", "method", "tile", "metric", "error"]].head(10))
    print(f"{len(failed):,} metric errors. Details in results.csv.")


def show_and_save(figure, name):
    path = report / f"{name}.html"
    config = {"displaylogo": False, "responsive": True}
    figure.write_html(str(path), include_plotlyjs=True, config=config)
    figure.show(renderer="plotly_mimetype+notebook", config=config)
    print(path)

## Mean Chamfer by tile position

Choose a method and image resolution in the dropdown. Each cell averages the
valid scores for that tile position across the matching sequences. Hover for the
actual 256 × 256 crop bounds and sample count. This is a schematic grid, so its
cells are drawn side by side even where the source crops overlap at the edges.

In [ ]:
# Use the same matched pairs in the map and method table.
used = {}
for metric in metrics:
    values = valid[valid.metric == metric]
    if MATCH_TILES and not values.empty:
        contributing = values.method.nunique()
        counts = values.groupby(["sequence", "tile"])["method"].transform("nunique")
        values = values[counts == contributing]
    used[metric] = values

geometry = ["source_width", "source_height"]
position = ["tile_row", "tile_column", "tile", "x0", "y0"]
layout = raw[geometry + position].drop_duplicates()
chamfer = used["chamfer_full"]
tile_means = (chamfer.groupby(["method", *geometry, *position], as_index=False)
              .agg(mean=("value", "mean"), n=("value", "size")))
tile_means.to_csv(report / "chamfer_tile_means.csv", index=False)

fig = go.Figure()
panels = []
max_mean = float(tile_means["mean"].max()) if not tile_means.empty else 1.0
for (width, height), grid in layout.groupby(geometry):
    nrows = int(grid.tile_row.max()) + 1
    ncols = int(grid.tile_column.max()) + 1
    for method in methods:
        means = tile_means[(tile_means.method == method) & (tile_means.source_width == width)
                           & (tile_means.source_height == height)]
        cells = grid.merge(means[position + ["mean", "n"]], on=position, how="left")
        z = np.full((nrows, ncols), np.nan)
        labels = np.full((nrows, ncols), "—", dtype=object)
        hover = np.full((nrows, ncols), "", dtype=object)
        for cell in cells.itertuples(index=False):
            r, c = cell.tile_row, cell.tile_column
            n = int(cell.n) if pd.notna(cell.n) else 0
            if n:
                z[r, c] = cell.mean
                labels[r, c] = f"{cell.mean:.3f}"
            hover[r, c] = (f"{cell.tile}<br>x [{cell.x0}, {cell.x0 + scoring.TILE_SIZE}) px"
                           f"<br>y [{cell.y0}, {cell.y0 + scoring.TILE_SIZE}) px"
                           f"<br>Mean Chamfer {labels[r, c]}<br>n = {n}")
        # Put a grey background behind the heatmap so missing cells are distinct from zero.
        shown = not panels
        fig.add_heatmap(z=np.zeros_like(z), showscale=False, hoverinfo="skip", visible=shown,
                        colorscale=[[0, "#e9edf1"], [1, "#e9edf1"]], xgap=3, ygap=3)
        fig.add_heatmap(z=z, coloraxis="coloraxis", xgap=3, ygap=3, visible=shown,
                        hoverinfo="skip", hoverongaps=False)
        rows, cols = np.indices(z.shape)
        fig.add_scatter(x=cols.ravel(), y=rows.ravel(), mode="markers+text",
                        text=labels.ravel(), customdata=hover.ravel(), textposition="middle center",
                        marker=dict(size=34, opacity=0), hovertemplate="%{customdata}<extra></extra>",
                        showlegend=False, visible=shown, textfont=dict(size=15, color="#14273b"))
        panels.append((f"{method} · {width} × {height}", nrows, ncols))

buttons = []
for i, (label, nr, nc) in enumerate(panels):
    visibility = [j // 3 == i for j in range(len(fig.data))]
    buttons.append(dict(label=label, method="update", args=[
        {"visible": visibility},
        {"title.text": f"{DATASET} · {label}", "xaxis.range": [-.5, nc-.5],
         "yaxis.range": [nr-.5, -.5]},
    ]))
first, nr, nc = panels[0]
fig.update_layout(template="plotly_white", height=max(390, 110 + 85 * max(p[1] for p in panels)),
                  title=f"{DATASET} · {first}", margin=dict(l=70, r=110, t=105, b=65),
                  coloraxis=dict(cmin=0, cmax=max(1e-9, max_mean),
                                 colorscale=[[0, "#fbfdff"], [.45, "#c6dcf2"], [1, "#6c9dc9"]],
                                 colorbar=dict(title="Chamfer")),
                  updatemenus=[dict(buttons=buttons, x=0, y=1.16, xanchor="left")])
fig.update_xaxes(title="Tile column", range=[-.5, nc-.5], dtick=1, showgrid=False)
fig.update_yaxes(title="Tile row", range=[nr-.5, -.5], dtick=1, showgrid=False,
                 scaleanchor="x", constrain="domain")
show_and_save(fig, "chamfer_tile_map")

## Mean metrics by method

Each valid sequence–tile pair contributes equally. **Coverage** shows the count
behind each average. With `MATCH_TILES=True`, the mean uses the intersection
of valid pairs among methods that have scores for that metric. Grey map cells
and dashes in the table indicate missing results. Chamfer requires both
polarities. Finite Fano scores with partial support are included.

In [ ]:
means = pd.DataFrame(index=pd.Index(methods, name="method"))
coverage = pd.DataFrame(index=means.index)
for metric, values in used.items():
    grouped = values.groupby("method")["value"]
    means[metric] = grouped.mean().reindex(means.index)
    coverage[metric] = grouped.size().reindex(means.index, fill_value=0)

means.to_csv(report / "method_means.csv")
coverage.to_csv(report / "method_coverage.csv")
print("Mean metrics")
display(HTML('<div style="overflow-x:auto">' + means.to_html(
    float_format=lambda value: f"{value:.5g}", na_rep="—") + "</div>"))
print("Coverage · scored sequence–tile pairs")
display(coverage)

## Best Chamfer case

By default this is the lowest valid **individual** `chamfer_full` score in
the selected dataset. The event-rate and ISI plots use the full sequence for
that tile. The space-time slices and 3D cloud use a short, configurable window.
Each chart is displayed here and saved as a standalone interactive HTML file.

In [ ]:
choices = valid[valid.metric == "chamfer_full"]
for column, selected in (("method", PLOT_METHOD), ("sequence", PLOT_SEQUENCE), ("tile", PLOT_TILE)):
    if selected is not None:
        choices = choices[choices[column] == selected]
best = choices.sort_values(["value", "method", "sequence", "tile"]).head(1)
if best.empty:
    print("No valid Chamfer case for these plot filters. Check coverage or clear the filters.")
else:
    display(best[["method", "sequence", "tile", "x0", "y0", "value"]].rename(
        columns={"value": "chamfer_full"}).reset_index(drop=True))

In [ ]:
def plot_isi(streams, meta, title, tag):
    intervals = {name: np.sort(scoring.bench.m.isi_distribution(events, meta["width"], meta["duration"]))
                 for name, events in streams.items()}
    upper = max([values[-1] for values in intervals.values() if values.size] + [.001])
    grid = np.r_[0, np.geomspace(1e-6, upper, 250)]
    fig = go.Figure()
    colors = ("#252b34", "#2563eb")
    for (name, values), color in zip(intervals.items(), colors):
        if values.size:
            fig.add_scatter(x=np.log10(1 + grid / .001), y=np.searchsorted(values, grid, side="right") / values.size,
                            customdata=grid, name=name, mode="lines", line=dict(color=color, width=2),
                            hovertemplate="%{customdata:.4g} ms · CDF %{y:.3f}<extra>%{fullData.name}</extra>")
    ticks = np.asarray([v for v in (0, .001, .01, .1, 1, 10, 100, 1000, 10000, 100000) if v <= upper])
    fig.update_layout(template="plotly_white", title=f"{title} · ISI CDF", height=470,
                      xaxis=dict(title="ISI (ms)", tickvals=np.log10(1 + ticks / .001),
                                 ticktext=[f"{v:g}" for v in ticks]),
                      yaxis=dict(title="Cumulative fraction", range=[0, 1.02]))
    if not fig.data:
        fig.add_annotation(text="No inter-event intervals in either stream", showarrow=False,
                           x=.5, y=.5, xref="paper", yref="paper")
    show_and_save(fig, f"{tag}_isi_cdf")


def clip_for_plots(streams, duration):
    if not np.isfinite(PLOT_WINDOW_MS) or PLOT_WINDOW_MS <= 0:
        raise ValueError("PLOT_WINDOW_MS must be positive.")
    span = PLOT_WINDOW_MS / 1000
    gt = streams["Ground truth"]
    if PLOT_START_MS is None:
        edges = scoring.voxels.time_edges(0., duration, span * 1000)
        start = float(edges[np.diff(np.searchsorted(gt["t"], edges)).argmax()])
    else:
        start = PLOT_START_MS / 1000
    stop = min(start + span, duration)
    if not np.isfinite(start) or not 0 <= start < stop:
        raise ValueError("Choose a plot start inside the sequence.")
    cut = {}
    for name, events in streams.items():
        a, b = np.searchsorted(events["t"], [start, stop])
        cut[name] = {key: value[a:b] for key, value in events.items()}
    return cut, start, stop


def display_indices(mask):
    idx = np.flatnonzero(mask)
    if idx.size > MAX_PLOT_POINTS:
        idx = idx[np.linspace(0, idx.size - 1, MAX_PLOT_POINTS, dtype=int)]
    return idx


def plot_spacetime(streams, meta, start, stop, title, tag):
    gt = streams["Ground truth"]
    sy = int(np.argmax(np.bincount(gt["y"], minlength=meta["height"])))
    sx = int(np.argmax(np.bincount(gt["x"], minlength=meta["width"])))
    fig = make_subplots(rows=1, cols=2, subplot_titles=(f"x–t at y={sy}", f"y–t at x={sx}"))
    cloud = go.Figure()
    for (name, events), color in zip(streams.items(), ("#4b5563", "#2563eb")):
        for polarity, symbol, positive in (("ON", "circle", True), ("OFF", "x", False)):
            sign = (events["p"] > 0) == positive
            label = f"{name} · {polarity}"
            for col, (other, pos, axis) in enumerate((("y", sy, "x"), ("x", sx, "y")), 1):
                idx = display_indices(sign & (events[other] == pos))
                fig.add_scattergl(x=events[axis][idx], y=1000 * events["t"][idx], name=label,
                                  mode="markers", legendgroup=label, showlegend=col == 1,
                                  marker=dict(color=color, symbol=symbol, size=6, opacity=.7), row=1, col=col)
            idx = display_indices(sign)
            cloud.add_scatter3d(x=events["x"][idx], y=1000 * events["t"][idx], z=events["y"][idx],
                                name=label, mode="markers",
                                marker=dict(color=color, symbol=symbol, size=2.3, opacity=.55))
    window = f"{1000 * start:.1f}–{1000 * stop:.1f} ms"
    fig.update_layout(template="plotly_white", height=510, title=f"{title} · {window}")
    fig.update_xaxes(title="x (px)", range=[0, meta["width"] - 1], row=1, col=1)
    fig.update_xaxes(title="y (px)", range=[0, meta["height"] - 1], row=1, col=2)
    fig.update_yaxes(title="Time (ms)", range=[1000 * start, 1000 * stop])
    cloud.update_layout(template="plotly_white", height=620, title=f"{title} · {window} · 3D events",
                        scene=dict(xaxis_title="x (px)", yaxis_title="Time (ms)", zaxis_title="y (px)",
                                   zaxis=dict(autorange="reversed"), aspectmode="cube"))
    show_and_save(fig, f"{tag}_spacetime")
    show_and_save(cloud, f"{tag}_cloud3d")


if not best.empty:
    row = best.iloc[0]
    gt, pred, meta = scoring.load_example(row)
    streams = {"Ground truth": gt, row["method"]: pred}
    title = (f"{DATASET}/{row['sequence']}/{row['tile']} · {row['method']}"
             f" · Chamfer {row['value']:.4g}")
    tag = f"{row['sequence']}_{row['tile']}_{row['method']}"
    duration = float(meta["duration"])
    if not np.isfinite(RATE_BIN_MS) or RATE_BIN_MS <= 0 or MAX_PLOT_POINTS < 1:
        raise ValueError("Choose a positive rate-bin width and point cap.")
    rate_bin_ms = max(RATE_BIN_MS, np.ceil(duration * 1000 / 5000))
    rate_onoff, rate_total, _ = rates.make_rate_figures(streams, 0, duration, rate_bin_ms,
                                                        f"{title} · {rate_bin_ms:g} ms bins")
    show_and_save(rate_onoff, f"{tag}_event_rate_on_off")
    show_and_save(rate_total, f"{tag}_event_rate_total")
    plot_isi(streams, meta, title, tag)
    cut, start, stop = clip_for_plots(streams, duration)
    plot_spacetime(cut, meta, start, stop, title, tag)